# Table Tennis Sportradar Data Review

This notebook reviews both:
- the saved `tabletennis/` raw dataset collected by `get_sportsradar_data.py`
- the `endpoint_probes/tabletennis/` outputs collected by `probe_tabletennis_endpoints.py`

Use it to inspect:
- competition / season / match coverage
- saved raw JSON layout
- flattened match-level tables
- endpoint probe runs and success rates
- what each endpoint stores and which fields appear in the payloads


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'tabletennis').exists() and (candidate / 'get_sportsradar_data.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the current notebook working directory.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'tabletennis'
RAW_ROOT = DATA_ROOT / 'raw'
METADATA_PATH = DATA_ROOT / 'metadata' / 'tabletennis_access_summary.json'
PROBE_ROOT = PROJECT_ROOT / 'endpoint_probes' / 'tabletennis'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('RAW_ROOT     =', RAW_ROOT)
print('METADATA     =', METADATA_PATH)
print('PROBE_ROOT   =', PROBE_ROOT)


PROJECT_ROOT = /data2/MHL/skill-vs-luck
DATA_ROOT    = /data2/MHL/skill-vs-luck/tabletennis
RAW_ROOT     = /data2/MHL/skill-vs-luck/tabletennis/raw
METADATA     = /data2/MHL/skill-vs-luck/tabletennis/metadata/tabletennis_access_summary.json
PROBE_ROOT   = /data2/MHL/skill-vs-luck/endpoint_probes/tabletennis


In [2]:
summary = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
details_df = pd.DataFrame(summary['details'])

headline = pd.Series({
    'saved_at_utc': summary['saved_at_utc'],
    'sport': summary['sport'],
    'competition_count': summary['competition_count'],
    'season_count': summary['season_count'],
    'match_count': summary['match_count'],
    'competition_rows_in_summary': len(details_df),
    'failed_seasons': len(summary.get('failed_seasons', [])),
    'errored_competitions': int(details_df['error'].notna().sum()) if 'error' in details_df else 0,
})

display(headline.to_frame('value'))
display(details_df.sort_values(['match_count', 'season_count'], ascending=False).head(15))


,value
saved_at_utc,2026-03-18T17:15:27.163423+00:00
sport,tabletennis
competition_count,470
season_count,622
match_count,23817
competition_rows_in_summary,470
failed_seasons,1
errored_competitions,190


,sport,competition_id,competition_name,season_count,match_count,season_ids,error
12,tabletennis,sr:competition:14447,"Champions League, Women",3,860,"[sr:season:110549, sr:season:123491, sr:season...",None
7,tabletennis,sr:competition:2302,Champions League,3,823,"[sr:season:110541, sr:season:125547, sr:season...",None
121,tabletennis,sr:competition:32749,ITTF European Championships Team Qualification,3,621,"[sr:season:86564, sr:season:110411, sr:season:...",None
133,tabletennis,sr:competition:35102,"European Team Championships Qualification, Women",3,563,"[sr:season:86568, sr:season:110413, sr:season:...",None
89,tabletennis,sr:competition:27322,"World Championships, MS",3,374,"[sr:season:105269, sr:season:105281, sr:season...",None
92,tabletennis,sr:competition:27328,"World Championships, WS",3,369,"[sr:season:105291, sr:season:105293, sr:season...",None
149,tabletennis,sr:competition:36339,WTT Singapore Smash,3,362,"[sr:season:116319, sr:season:128191, sr:season...",None
151,tabletennis,sr:competition:36343,"WTT Singapore Smash, Women",3,360,"[sr:season:116325, sr:season:128197, sr:season...",None
4,tabletennis,sr:competition:1858,Bundesliga,3,350,"[sr:season:45060, sr:season:57478, sr:season:7...",None
241,tabletennis,sr:competition:40273,"WTT Star Contender Ljubljana, MS",3,304,"[sr:season:107233, sr:season:119443, sr:season...",None


In [3]:
competition_files = sorted((RAW_ROOT / 'competitions').glob('*/seasons.json'))
season_summary_files = sorted((RAW_ROOT / 'seasons').glob('*/summaries.json'))
raw_competitions_payload = json.loads((RAW_ROOT / 'competitions.json').read_text(encoding='utf-8'))
raw_competitions = raw_competitions_payload.get('competitions', [])


def decode_saved_resource_name(name: str) -> str:
    parts = name.split('_', 2)
    if len(parts) == 3:
        return f'{parts[0]}:{parts[1]}:{parts[2]}'
    return name


saved_season_ids = {decode_saved_resource_name(path.parent.name) for path in season_summary_files}
season_ids_from_summary = {season_id for row in summary['details'] for season_id in row.get('season_ids', [])}
missing_summary_files = sorted(season_ids_from_summary - saved_season_ids)

layout_df = pd.DataFrame([
    {'item': 'raw_competitions_json_exists', 'value': (RAW_ROOT / 'competitions.json').exists()},
    {'item': 'competition_season_files', 'value': len(competition_files)},
    {'item': 'season_summary_files', 'value': len(season_summary_files)},
    {'item': 'competitions_in_raw_payload', 'value': len(raw_competitions)},
    {'item': 'season_ids_in_metadata', 'value': len(season_ids_from_summary)},
    {'item': 'missing_summary_files', 'value': len(missing_summary_files)},
])

display(layout_df)
display(pd.Series(missing_summary_files[:20], name='first_missing_season_ids'))


,item,value
0,raw_competitions_json_exists,True
1,competition_season_files,280
2,season_summary_files,621
3,competitions_in_raw_payload,470
4,season_ids_in_metadata,622
5,missing_summary_files,1


0    sr:season:123195
Name: first_missing_season_ids, dtype: object

In [4]:
competitions_df = pd.json_normalize(raw_competitions, sep='_')
display(competitions_df.head())

season_rows = []
for path in competition_files:
    payload = json.loads(path.read_text(encoding='utf-8'))
    for season in payload.get('seasons', []):
        season_rows.append({
            'source_file': str(path.relative_to(PROJECT_ROOT)),
            **season,
        })

seasons_df = pd.DataFrame(season_rows)
if not seasons_df.empty:
    seasons_df['start_date'] = pd.to_datetime(seasons_df['start_date'], errors='coerce')
    seasons_df['end_date'] = pd.to_datetime(seasons_df['end_date'], errors='coerce')

display(seasons_df.head())
display(seasons_df[['competition_id', 'year']].value_counts().head(15).rename('season_rows'))


,id,name,type,gender,category_id,category_name,category_country_code,parent_id
0,sr:competition:564,Olympic Tournament,singles,men,sr:category:88,International,NaN,NaN
1,sr:competition:565,Olympic Tournament Women,singles,women,sr:category:88,International,NaN,NaN
2,sr:competition:1121,"Olympic Tournament, Team",singles,men,sr:category:88,International,NaN,NaN
3,sr:competition:1123,Olympic Tournament Women,singles,women,sr:category:88,International,NaN,NaN
4,sr:competition:1858,Bundesliga,singles,men,sr:category:145,Germany,DEU,NaN


,source_file,id,name,start_date,end_date,year,competition_id
0,tabletennis/raw/competitions/sr_competition_11...,sr:season:34305,"Olympic Tournament, Team 2016",2016-08-12,2016-08-19,2016,sr:competition:1121
1,tabletennis/raw/competitions/sr_competition_11...,sr:season:84982,"Olympic Tournament 2020, Team",2021-08-01,2021-08-06,2021,sr:competition:1121
2,tabletennis/raw/competitions/sr_competition_11...,sr:season:105601,"Olympic Tournament, Team 2024",2024-07-26,2024-08-09,2024,sr:competition:1121
3,tabletennis/raw/competitions/sr_competition_11...,sr:season:34307,Olympic Tournament Women 2016,2016-08-12,2016-08-19,2016,sr:competition:1123
4,tabletennis/raw/competitions/sr_competition_11...,sr:season:84980,"Olympic Tournament Women 2020, Team",2021-08-01,2021-08-06,2021,sr:competition:1123


competition_id        year
sr:competition:14107  2026    3
sr:competition:23053  2024    2
sr:competition:23059  2024    2
sr:competition:37161  2025    1
sr:competition:37157  2024    1
                      2025    1
                      2026    1
sr:competition:37159  2024    1
                      2025    1
                      2026    1
sr:competition:37161  2024    1
sr:competition:1121   2016    1
sr:competition:37037  2024    1
sr:competition:37163  2024    1
                      2025    1
Name: season_rows, dtype: int64

In [5]:
sample_summary_path = season_summary_files[0]
sample_summary_payload = json.loads(sample_summary_path.read_text(encoding='utf-8'))

print('sample_summary_path =', sample_summary_path.relative_to(PROJECT_ROOT))
print('top_level_keys =', list(sample_summary_payload.keys()))
print('summary_count =', len(sample_summary_payload.get('summaries', [])))

sample_rows = pd.json_normalize(sample_summary_payload.get('summaries', [])[:3], sep='.')
display(sample_rows)


sample_summary_path = tabletennis/raw/seasons/sr_season_101759/summaries.json
top_level_keys = ['generated_at', 'summaries']
summary_count = 31


,sport_event.id,sport_event.start_time,sport_event.start_time_confirmed,sport_event.sport_event_context.sport.id,sport_event.sport_event_context.sport.name,sport_event.sport_event_context.category.id,sport_event.sport_event_context.category.name,sport_event.sport_event_context.competition.id,sport_event.sport_event_context.competition.name,sport_event.sport_event_context.competition.type,sport_event.sport_event_context.competition.gender,sport_event.sport_event_context.season.id,sport_event.sport_event_context.season.name,sport_event.sport_event_context.season.start_date,sport_event.sport_event_context.season.end_date,sport_event.sport_event_context.season.year,sport_event.sport_event_context.season.competition_id,sport_event.sport_event_context.stage.order,sport_event.sport_event_context.stage.type,sport_event.sport_event_context.stage.phase,sport_event.sport_event_context.stage.start_date,sport_event.sport_event_context.stage.end_date,sport_event.sport_event_context.stage.year,sport_event.sport_event_context.round.name,sport_event.sport_event_context.groups,sport_event.sport_event_context.mode.best_of,sport_event.coverage.live,sport_event.competitors,sport_event_status.status,sport_event_status.match_status,sport_event_status.home_score,sport_event_status.away_score,sport_event_status.winner_id,sport_event_status.period_scores
0,sr:sport_event:38577579,2023-01-12T09:10:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023-01-10,2023-01-15,2023,sr:competition:38979,1,cup,playoffs,2023-01-10,2023-01-15,2023,round_of_32,"[{'id': 'sr:cup:142837', 'name': 'WTT Contende...",5,True,"[{'id': 'sr:competitor:978497', 'name': 'Mooke...",closed,ended,0,3,sr:competitor:76859,"[{'home_score': 2, 'away_score': 11, 'type': '..."
1,sr:sport_event:38577595,2023-01-12T09:10:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023-01-10,2023-01-15,2023,sr:competition:38979,1,cup,playoffs,2023-01-10,2023-01-15,2023,round_of_32,"[{'id': 'sr:cup:142837', 'name': 'WTT Contende...",5,True,"[{'id': 'sr:competitor:394314', 'name': 'Surav...",closed,ended,3,0,sr:competitor:394314,"[{'home_score': 11, 'away_score': 5, 'type': '..."
2,sr:sport_event:38577581,2023-01-12T09:45:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023-01-10,2023-01-15,2023,sr:competition:38979,1,cup,playoffs,2023-01-10,2023-01-15,2023,round_of_32,"[{'id': 'sr:cup:142837', 'name': 'WTT Contende...",5,True,"[{'id': 'sr:competitor:933143', 'name': 'Liang...",closed,ended,1,3,sr:competitor:389452,"[{'home_score': 13, 'away_score': 11, 'type': ..."


In [6]:
sample_rows['sport_event.competitors'].iloc[0]

[{'id': 'sr:competitor:978497',
  'name': 'Mooketsi, Ndelu',
  'country': 'Botswana',
  'country_code': 'BWA',
  'abbreviation': 'MOO',
  'qualifier': 'home',
  'gender': 'male',
  'date_of_birth': '1998-07-10'},
 {'id': 'sr:competitor:76859',
  'name': 'Zhmudenko, Yaroslav',
  'country': 'Ukraine',
  'country_code': 'UKR',
  'abbreviation': 'ZHM',
  'qualifier': 'away',
  'gender': 'male',
  'date_of_birth': '1988-09-24'}]

In [7]:
sample_rows['sport_event_status.period_scores'].iloc[0]

[{'home_score': 2, 'away_score': 11, 'type': 'regular_period', 'number': 1},
 {'home_score': 3, 'away_score': 11, 'type': 'regular_period', 'number': 2},
 {'home_score': 2, 'away_score': 11, 'type': 'regular_period', 'number': 3}]

In [8]:
def pick_competitor_name(competitors, qualifier):
    for competitor in competitors or []:
        if competitor.get('qualifier') == qualifier:
            return competitor.get('name')
    return None


match_rows = []
for path in season_summary_files:
    payload = json.loads(path.read_text(encoding='utf-8'))
    for item in payload.get('summaries', []):
        sport_event = item.get('sport_event', {})
        event_status = item.get('sport_event_status', {})
        context = sport_event.get('sport_event_context', {})
        competition = context.get('competition', {})
        season = context.get('season', {})
        category = context.get('category', {})
        stage = context.get('stage', {})
        round_info = context.get('round', {})
        mode = context.get('mode', {})
        competitors = sport_event.get('competitors', [])

        match_rows.append({
            'sport_event_id': sport_event.get('id'),
            'start_time': sport_event.get('start_time'),
            'start_time_confirmed': sport_event.get('start_time_confirmed'),
            'competition_id': competition.get('id'),
            'competition_name': competition.get('name'),
            'competition_type': competition.get('type'),
            'competition_gender': competition.get('gender'),
            'season_id': season.get('id'),
            'season_name': season.get('name'),
            'season_year': season.get('year'),
            'category_id': category.get('id'),
            'category_name': category.get('name'),
            'stage_type': stage.get('type'),
            'stage_phase': stage.get('phase'),
            'round_name': round_info.get('name'),
            'best_of': mode.get('best_of'),
            'status': event_status.get('status'),
            'match_status': event_status.get('match_status'),
            'home_score': event_status.get('home_score'),
            'away_score': event_status.get('away_score'),
            'winner_id': event_status.get('winner_id'),
            'period_count': len(event_status.get('period_scores', []) or []),
            'home_player': pick_competitor_name(competitors, 'home'),
            'away_player': pick_competitor_name(competitors, 'away'),
            'source_file': str(path.relative_to(PROJECT_ROOT)),
        })

matches_df = pd.DataFrame(match_rows)
if not matches_df.empty:
    matches_df['start_time'] = pd.to_datetime(matches_df['start_time'], errors='coerce', utc=True)

print('matches_df shape =', matches_df.shape)
display(matches_df.head())


matches_df shape = (24171, 25)


,sport_event_id,start_time,start_time_confirmed,competition_id,competition_name,competition_type,competition_gender,season_id,season_name,season_year,category_id,category_name,stage_type,stage_phase,round_name,best_of,status,match_status,home_score,away_score,winner_id,period_count,home_player,away_player,source_file
0,sr:sport_event:38577579,2023-01-12 09:10:00+00:00,True,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023,sr:category:88,International,cup,playoffs,round_of_32,5.0,closed,ended,0.0,3.0,sr:competitor:76859,3,"Mooketsi, Ndelu","Zhmudenko, Yaroslav",tabletennis/raw/seasons/sr_season_101759/summa...
1,sr:sport_event:38577595,2023-01-12 09:10:00+00:00,True,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023,sr:category:88,International,cup,playoffs,round_of_32,5.0,closed,ended,3.0,0.0,sr:competitor:394314,3,"Suravajjula, Snehit","Sapire, Tevia",tabletennis/raw/seasons/sr_season_101759/summa...
2,sr:sport_event:38577581,2023-01-12 09:45:00+00:00,True,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023,sr:category:88,International,cup,playoffs,round_of_32,5.0,closed,ended,1.0,3.0,sr:competitor:389452,4,"Liang, Yanning","Rolland, Jules",tabletennis/raw/seasons/sr_season_101759/summa...
3,sr:sport_event:38577593,2023-01-12 09:45:00+00:00,True,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023,sr:category:88,International,cup,playoffs,round_of_32,5.0,closed,ended,3.0,0.0,sr:competitor:304506,3,"Jarvis, Thomas David","Xue, Fei",tabletennis/raw/seasons/sr_season_101759/summa...
4,sr:sport_event:38577571,2023-01-12 12:00:00+00:00,True,sr:competition:38979,"WTT Contender Durban, Singles",singles,men,sr:season:101759,"WTT Contender Durban, Singles 2023",2023,sr:category:88,International,cup,playoffs,round_of_32,5.0,closed,ended,3.0,0.0,sr:competitor:303490,3,"Xu, Yingbin","Glod, Eric",tabletennis/raw/seasons/sr_season_101759/summa...


In [9]:
quality_checks = pd.DataFrame([
    {'check': 'unique sport_event_id', 'value': int(matches_df['sport_event_id'].nunique())},
    {'check': 'duplicate sport_event_id rows', 'value': int(matches_df['sport_event_id'].duplicated().sum())},
    {'check': 'missing start_time', 'value': int(matches_df['start_time'].isna().sum())},
    {'check': 'missing competition_name', 'value': int(matches_df['competition_name'].isna().sum())},
    {'check': 'missing home_player', 'value': int(matches_df['home_player'].isna().sum())},
    {'check': 'missing away_player', 'value': int(matches_df['away_player'].isna().sum())},
])

display(quality_checks)
display(matches_df['status'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
display(matches_df['match_status'].value_counts(dropna=False).rename_axis('match_status').to_frame('rows'))
display(matches_df['round_name'].value_counts(dropna=False).head(20).rename_axis('round_name').to_frame('rows'))
display(matches_df.groupby('competition_name').size().sort_values(ascending=False).head(20).rename('matches'))


,check,value
0,unique sport_event_id,23809
1,duplicate sport_event_id rows,362
2,missing start_time,0
3,missing competition_name,0
4,missing home_player,1
5,missing away_player,1


,rows
status,
closed,23305
cancelled,519
ended,297
not_started,46
postponed,4


,rows
match_status,
ended,23581
None,519
not_started,41
walkover,17
retired,7
postponed,4
3rd_set,1
pause,1


,rows
round_name,
None,7384
round_of_16,4488
round_of_32,4317
round_of_64,2634
quarterfinal,2498
semifinal,1201
final,563
round_of_128,497
round_1,235


competition_name
Champions League, Women                             860
Champions League                                    823
ITTF European Championships Team Qualification      602
European Team Championships Qualification, Women    558
Olympic Tournament Women                            391
World Championships, MS                             374
World Championships, WS                             369
WTT Singapore Smash                                 362
WTT Singapore Smash, Women                          360
Bundesliga                                          350
WTT Star Contender Ljubljana, MS                    304
WTT Feeder Dusseldorf, MS                           298
WTT Star Contender Ljubljana, WS                    296
WTT Contender Zagreb                                290
WTT Star Contender Doha, Singles                    288
WTT Feeder Otocec, MS                               274
WTT Contender Muscat                                266
WTT Contender Zagreb, Women    

## Endpoint Probe Review

The cells below inspect the outputs saved by `probe_tabletennis_endpoints.py`.
They let you check not only whether an endpoint succeeded, but also what keys and record-level fields are stored in each endpoint payload.


In [10]:
run_summary_paths = sorted(PROBE_ROOT.glob('*/probe_summary.json'))
run_rows = []

for path in run_summary_paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    run_rows.append({
        'run_name': path.parent.name,
        'saved_at_utc': payload.get('saved_at_utc'),
        'job_count': payload.get('job_count'),
        'success_count': payload.get('success_count'),
        'failure_count': payload.get('failure_count'),
        'run_dir': payload.get('run_dir'),
    })

runs_df = pd.DataFrame(run_rows).sort_values('saved_at_utc', ascending=False)
display(runs_df)


,run_name,saved_at_utc,job_count,success_count,failure_count,run_dir
0,20260324T104302Z,2026-03-24T10:44:34.140244+00:00,40,40,0,/data2/MHL/skill-vs-luck/endpoint_probes/table...


In [11]:
successful_runs = runs_df[runs_df['success_count'] > 0]
if not successful_runs.empty:
    selected_run_name = successful_runs.iloc[0]['run_name']
else:
    selected_run_name = runs_df.iloc[0]['run_name']

selected_run_dir = PROBE_ROOT / selected_run_name
selected_summary = json.loads((selected_run_dir / 'probe_summary.json').read_text(encoding='utf-8'))
selected_context = selected_summary['sample_context']
results_df = pd.DataFrame(selected_summary['results'])

results_df['started_at_utc'] = pd.to_datetime(results_df['started_at_utc'], utc=True)
results_df['finished_at_utc'] = pd.to_datetime(results_df['finished_at_utc'], utc=True)
results_df['duration_sec'] = (results_df['finished_at_utc'] - results_df['started_at_utc']).dt.total_seconds()

print('selected_run_name =', selected_run_name)
display(pd.Series({
    'job_count': selected_summary['job_count'],
    'success_count': selected_summary['success_count'],
    'failure_count': selected_summary['failure_count'],
    'run_dir': selected_summary['run_dir'],
}).to_frame('value'))
display(pd.DataFrame([selected_context]))
display(results_df[['endpoint', 'resource', 'status', 'duration_sec', 'saved_path']].sort_values(['status', 'endpoint', 'resource']))


selected_run_name = 20260324T104302Z


,value
job_count,40
success_count,40
failure_count,0
run_dir,/data2/MHL/skill-vs-luck/endpoint_probes/table...


,competition_ids,season_ids,match_samples,competitor_ids,versus_pair,date_strs
0,"[sr:competition:564, sr:competition:565]","[sr:season:34299, sr:season:84976]","[{'sport_event_id': 'sr:sport_event:38577579',...","[sr:competitor:978497, sr:competitor:76859, sr...","[sr:competitor:978497, sr:competitor:76859]",[2023-01-12]


,endpoint,resource,status,duration_sec,saved_path
10,competition_info,sr:competition:564,success,1.951620,/data2/MHL/skill-vs-luck/endpoint_probes/table...
12,competition_info,sr:competition:565,success,1.952407,/data2/MHL/skill-vs-luck/endpoint_probes/table...
11,competition_seasons,sr:competition:564,success,1.924799,/data2/MHL/skill-vs-luck/endpoint_probes/table...
13,competition_seasons,sr:competition:565,success,1.954292,/data2/MHL/skill-vs-luck/endpoint_probes/table...
0,competitions,all,success,2.388371,/data2/MHL/skill-vs-luck/endpoint_probes/table...
3,competitor_merge_mappings,all,success,1.892330,/data2/MHL/skill-vs-luck/endpoint_probes/table...
37,competitor_profile,sr:competitor:394314,success,2.010337,/data2/MHL/skill-vs-luck/endpoint_probes/table...
35,competitor_profile,sr:competitor:76859,success,2.033998,/data2/MHL/skill-vs-luck/endpoint_probes/table...
33,competitor_profile,sr:competitor:978497,success,1.975434,/data2/MHL/skill-vs-luck/endpoint_probes/table...
38,competitor_summaries,sr:competitor:394314,success,3.004146,/data2/MHL/skill-vs-luck/endpoint_probes/table...


In [12]:
def describe_payload(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = {
        'saved_path': str(path.relative_to(PROJECT_ROOT)),
        'top_level_keys': ', '.join(payload.keys()),
        'primary_collection': None,
        'primary_count': None,
        'sample_fields': None,
    }

    for key, value in payload.items():
        if isinstance(value, list):
            row['primary_collection'] = key
            row['primary_count'] = len(value)
            if value and isinstance(value[0], dict):
                row['sample_fields'] = ', '.join(value[0].keys())
            break
        if isinstance(value, dict):
            row['primary_collection'] = key
            row['primary_count'] = len(value)
            row['sample_fields'] = ', '.join(value.keys())
            break

    return row


endpoint_detail_rows = []
for _, row in results_df[results_df['status'] == 'success'].iterrows():
    endpoint_detail_rows.append({
        'endpoint': row['endpoint'],
        'resource': row['resource'],
        'duration_sec': row['duration_sec'],
        **describe_payload(Path(row['saved_path'])),
    })

endpoint_details_df = pd.DataFrame(endpoint_detail_rows).sort_values(['endpoint', 'resource'])
display(endpoint_details_df)


,endpoint,resource,duration_sec,saved_path,top_level_keys,primary_collection,primary_count,sample_fields
10,competition_info,sr:competition:564,1.951620,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competition",competition,5,"id, name, type, gender, category"
12,competition_info,sr:competition:565,1.952407,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competition",competition,5,"id, name, type, gender, category"
11,competition_seasons,sr:competition:564,1.924799,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, seasons",seasons,3,"id, name, start_date, end_date, year, competit..."
13,competition_seasons,sr:competition:565,1.954292,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, seasons",seasons,3,"id, name, start_date, end_date, year, competit..."
0,competitions,all,2.388371,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competitions",competitions,470,"id, name, type, gender, category"
3,competitor_merge_mappings,all,1.892330,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, mappings",mappings,0,None
37,competitor_profile,sr:competitor:394314,2.010337,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competitor, category, sport",competitor,7,"id, name, country, country_code, abbreviation,..."
35,competitor_profile,sr:competitor:76859,2.033998,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competitor, category, sport",competitor,7,"id, name, country, country_code, abbreviation,..."
33,competitor_profile,sr:competitor:978497,1.975434,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, competitor, category, sport",competitor,7,"id, name, country, country_code, abbreviation,..."
38,competitor_summaries,sr:competitor:394314,3.004146,endpoint_probes/tabletennis/20260324T104302Z/c...,"generated_at, summaries",summaries,30,"sport_event, sport_event_status"


In [13]:
def load_payload_preview(endpoint_name: str):
    subset = results_df[(results_df['endpoint'] == endpoint_name) & (results_df['status'] == 'success')]
    if subset.empty:
        print(f'No successful payload for endpoint: {endpoint_name}')
        return None

    payload_path = Path(subset.iloc[0]['saved_path'])
    payload = json.loads(payload_path.read_text(encoding='utf-8'))
    print('endpoint =', endpoint_name)
    print('payload_path =', payload_path.relative_to(PROJECT_ROOT))
    print('top_level_keys =', list(payload.keys()))

    for key, value in payload.items():
        if isinstance(value, list):
            print(f'{key}_count =', len(value))
            if value and isinstance(value[0], dict):
                display(pd.json_normalize(value[:3], sep='.'))
            break
        if isinstance(value, dict):
            display(pd.json_normalize([value], sep='.'))
            break

    total_df = pd.json_normalize(value, sep='.')
    return total_df


for endpoint_name in endpoint_details_df['endpoint'].unique():
    _ = load_payload_preview(endpoint_name)


endpoint = competition_info
payload_path = endpoint_probes/tabletennis/20260324T104302Z/competition_info/sr_competition_564.json
top_level_keys = ['generated_at', 'competition']


,id,name,type,gender,category.id,category.name
0,sr:competition:564,Olympic Tournament,singles,men,sr:category:88,International


endpoint = competition_seasons
payload_path = endpoint_probes/tabletennis/20260324T104302Z/competition_seasons/sr_competition_564.json
top_level_keys = ['generated_at', 'seasons']
seasons_count = 3


,id,name,start_date,end_date,year,competition_id
0,sr:season:34299,Olympic Games 2016,2016-08-06,2016-08-19,2016,sr:competition:564
1,sr:season:84976,Olympic Tournament 2020,2021-07-24,2021-07-30,2021,sr:competition:564
2,sr:season:105599,Olympic Games 2024,2024-07-27,2024-08-04,2024,sr:competition:564


endpoint = competitions
payload_path = endpoint_probes/tabletennis/20260324T104302Z/competitions/all.json
top_level_keys = ['generated_at', 'competitions']
competitions_count = 470


...

,id,active_season,created_at
0,sr:sport_event:70185208,True,2026-03-23T15:06:50+00:00
1,sr:sport_event:70185210,True,2026-03-23T15:06:51+00:00
2,sr:sport_event:70185212,True,2026-03-23T15:06:51+00:00


endpoint = sport_events_removed
payload_path = endpoint_probes/tabletennis/20260324T104302Z/sport_events_removed/all.json
top_level_keys = ['generated_at', 'sport_events_removed']
sport_events_removed_count = 200


,id
0,sr:sport_event:69860896
1,sr:sport_event:69860898
2,sr:sport_event:70001810


endpoint = sport_events_updated
payload_path = endpoint_probes/tabletennis/20260324T104302Z/sport_events_updated/all.json
top_level_keys = ['generated_at', 'sport_events_updated']
sport_events_updated_count = 102


,id,updated_at
0,sr:sport_event:70226008,2026-03-24T10:43:14+00:00
1,sr:sport_event:70185224,2026-03-24T10:43:03+00:00
2,sr:sport_event:70185256,2026-03-24T10:42:52+00:00


In [14]:
rankings_df = load_payload_preview("rankings")

endpoint = rankings
payload_path = endpoint_probes/tabletennis/20260324T104302Z/rankings/all.json
top_level_keys = ['generated_at', 'rankings']
rankings_count = 8


,type_id,name,year,week,gender,competitor_rankings
0,61,ittf_men_singles_world_ranking,2026,13,men,"[{'rank': 1, 'points': 10750, 'competitor': {'..."
1,62,ittf_men_doubles_pairs_world_ranking,2026,13,men,"[{'rank': 1, 'points': 5163, 'competitor': {'i..."
2,63,ittf_men_doubles_individual_world_ranking,2026,13,men,"[{'rank': 1, 'points': 5905, 'competitor': {'i..."


In [15]:
rankings_df['competitor_rankings'].iloc[0][0]

{'rank': 1,
 'points': 10750,
 'competitor': {'id': 'sr:competitor:299060',
  'name': 'Wang, Chuqin',
  'country': 'China',
  'country_code': 'CHN',
  'abbreviation': 'WAN',
  'gender': 'male',
  'date_of_birth': '2000-05-11'}}

In [16]:
season_probabilities_df = load_payload_preview("season_probabilities")

endpoint = season_probabilities
payload_path = endpoint_probes/tabletennis/20260324T104302Z/season_probabilities/sr_season_34299.json
top_level_keys = ['generated_at', 'sport_event_probabilities']
sport_event_probabilities_count = 70


,markets,sport_event.id,sport_event.start_time,sport_event.start_time_confirmed,sport_event.sport_event_context.sport.id,sport_event.sport_event_context.sport.name,sport_event.sport_event_context.category.id,sport_event.sport_event_context.category.name,sport_event.sport_event_context.competition.id,sport_event.sport_event_context.competition.name,sport_event.sport_event_context.competition.type,sport_event.sport_event_context.competition.gender,sport_event.sport_event_context.season.id,sport_event.sport_event_context.season.name,sport_event.sport_event_context.season.start_date,sport_event.sport_event_context.season.end_date,sport_event.sport_event_context.season.year,sport_event.sport_event_context.season.competition_id,sport_event.sport_event_context.stage.order,sport_event.sport_event_context.stage.type,sport_event.sport_event_context.stage.phase,sport_event.sport_event_context.stage.start_date,sport_event.sport_event_context.stage.end_date,sport_event.sport_event_context.stage.year,sport_event.sport_event_context.round.number,sport_event.sport_event_context.round.name,sport_event.sport_event_context.groups,sport_event.sport_event_context.mode.best_of,sport_event.coverage.live,sport_event.competitors
0,"[{'name': '2way', 'outcomes': [{'name': 'home_...",sr:sport_event:9905303,2016-08-06T12:45:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:564,Olympic Tournament,singles,men,sr:season:34299,Olympic Games 2016,2016-08-06,2016-08-19,2016,sr:competition:564,1,cup,stage_1_playoff,2016-08-06,2016-08-19,2016,1,qualification_round_1,"[{'id': 'sr:cup:63313', 'name': 'Olympic Games...",7,True,"[{'id': 'sr:competitor:276369', 'name': 'Afana..."
1,"[{'name': '2way', 'outcomes': [{'name': 'home_...",sr:sport_event:9905305,2016-08-06T12:45:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:564,Olympic Tournament,singles,men,sr:season:34299,Olympic Games 2016,2016-08-06,2016-08-19,2016,sr:competition:564,1,cup,stage_1_playoff,2016-08-06,2016-08-19,2016,1,qualification_round_1,"[{'id': 'sr:cup:63313', 'name': 'Olympic Games...",7,True,"[{'id': 'sr:competitor:24882', 'name': 'Aguirr..."
2,"[{'name': '2way', 'outcomes': [{'name': 'home_...",sr:sport_event:9905307,2016-08-06T13:30:00+00:00,True,sr:sport:20,Table Tennis,sr:category:88,International,sr:competition:564,Olympic Tournament,singles,men,sr:season:34299,Olympic Games 2016,2016-08-06,2016-08-19,2016,sr:competition:564,1,cup,stage_1_playoff,2016-08-06,2016-08-19,2016,1,qualification_round_1,"[{'id': 'sr:cup:63313', 'name': 'Olympic Games...",7,True,"[{'id': 'sr:competitor:202103', 'name': 'Alami..."


In [17]:
season_probabilities_df['markets'].iloc[0]

[{'name': '2way',
  'outcomes': [{'name': 'home_team_winner', 'probability': 72.9},
   {'name': 'away_team_winner', 'probability': 27.1}]}]